In [ ]:
# !pip install duckdb openai pydantic rich plotly -q

In [ ]:
# import duckdb
# conn = duckdb.connect("../data/AdventureWorks.duckdb")
# # conn.execute("SHOW TABLES").fetchdf()
# print(conn)

# conn.execute("PRAGMA database_list").fetchdf()
# conn.execute("SHOW TABLES").fetchdf()

In [ ]:
import duckdb
import pandas as pd

# Connect to in-memory DuckDB
conn = duckdb.connect(":memory:")

# Load your CSV
CSV_PATH = "../data/credit.csv"  # ← change this
conn.execute(f"CREATE TABLE data AS SELECT * FROM read_csv_auto('{CSV_PATH}')")

# Inspect — always do this first so you know what you're working with
print("=== SCHEMA ===")
print(conn.execute("DESCRIBE data").fetchdf().to_string(index=False))

print("\n=== ROW COUNT ===")
print(conn.execute("SELECT COUNT(*) as total_rows FROM data").fetchone()[0])

print("\n=== SAMPLE (5 rows) ===")
print(conn.execute("SELECT * FROM data LIMIT 5").fetchdf().to_string(index=False))

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# ── Model 1: Tool Input ──────────────────────────────────────────
# Validates what the agent sends TO the tool before it hits DuckDB
class RunSqlInput(BaseModel):
    sql: str = Field(..., description="Valid DuckDB SQL query")

# ── Model 2: Agent Final Answer ──────────────────────────────────
# Enforces the shape of what the agent returns TO you
class AgentAnswer(BaseModel):
    answer: str = Field(..., description="Plain English explanation of the result")
    sql_used: str = Field(..., description="The SQL that produced the final result")
    chart_type: Literal["bar", "line", "pie", "scatter", "histogram", "none"]
    plotly_code: str = Field(..., description="Complete plotly figure code using df and fig variables")

print("✅ Models defined")
print(f"   RunSqlInput fields : {list(RunSqlInput.model_fields.keys())}")
print(f"   AgentAnswer fields : {list(AgentAnswer.model_fields.keys())}")


In [ ]:
# These are declarations — they tell the LLM what tools exist
# and what inputs they expect. No actual logic here.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": "Get column names and types from the database. Always call this first before writing any SQL.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_sql",
            "description": "Run a SQL query on the DuckDB table called 'data'. Use this to answer questions about the data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "sql": {
                        "type": "string",
                        "description": "Valid DuckDB SQL query. Always LIMIT to 100 rows max."
                    }
                },
                "required": ["sql"]
            }
        }
    }
]

print("✅ Tools defined")
for t in TOOLS:
    print(f"   → {t['function']['name']}: {t['function']['description'][:60]}...")

In [ ]:
import json

def execute_tool(tool_name: str, tool_args: dict) -> str:
    """
    Receives tool call from agent, runs it, returns result as string.
    All tool results must be strings — that's what the LLM can read.
    """

    if tool_name == "get_schema":
        result = conn.execute("DESCRIBE data").fetchdf()
        return result.to_string(index=False)

    elif tool_name == "run_sql":
        # Validate input with Pydantic before touching the DB
        args = RunSqlInput(**tool_args)
        try:
            df = conn.execute(args.sql).fetchdf()
            return df.to_string(index=False)
        except Exception as e:
            # Return error as string so agent can self-correct
            return f"SQL_ERROR: {str(e)}"

    return f"UNKNOWN_TOOL: {tool_name}"


print("✅ Tool executor defined")
print("\n--- Quick test ---")
print(execute_tool("get_schema", {}))

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

SYSTEM_PROMPT = """You are a data analyst agent. The database has one table called 'data'.

Follow this exact process:
1. Call get_schema to understand the data
2. Call run_sql to query the data
3. Repeat step 2 if needed
4. Return your final structured answer

SQL rules:
- If fetching raw rows → always LIMIT 100
- If using GROUP BY, COUNT, SUM, AVG etc → NO LIMIT, you need all groups for accurate charts
- Always aggregate in SQL, never return raw data for charts

For plotly_code:
- Use plotly.express as px or plotly.graph_objects as go
- Data is in a pandas DataFrame called df (your last query result)
- Create a figure called fig
- Do NOT call fig.show()
"""

def run_agent(question: str) -> AgentAnswer:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question}
    ]

    print(f"Question: {question}\n")

    # ── Loop until agent stops calling tools ─────────────────────
    step = 0
    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)                    # add assistant turn to history

        # ── No tool calls = agent is done thinking ────────────────
        if not msg.tool_calls:
            break
        # ── Execute each tool the agent requested ─────────────────
        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)
            print(f"Step {step}: {tc.function.name}({args})")

            result = execute_tool(tc.function.name, args)

            # Feed result back into conversation so agent can see it
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })

    # ── Get final structured answer ───────────────────────────────
    final = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=messages + [{
            "role": "user",
            "content": "Now give your final structured answer."
        }],
        response_format=AgentAnswer
    )

    return final.choices[0].message.parsed

In [ ]:
# ── Test ──────────────────────────────────────────────────────────
result = run_agent("give me a graph between sex and credit history")
print(f"\nAnswer: {result.answer}")
print(f"Chart:  {result.chart_type}")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

def run_agent_with_chart(question: str) -> AgentAnswer:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question}
    ]

    console.rule("[bold blue]New Question")
    console.print(f"[bold white]Q: {question}\n")

    step = 0
    last_df = None  # ← stores the last SQL result as DataFrame

    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "get_schema":
                console.print(f"[cyan]Step {step}:[/] get_schema()")
                result = execute_tool("get_schema", {})

            elif tc.function.name == "run_sql":
                console.print(f"[cyan]Step {step}:[/] run_sql()")
                console.print(Syntax(args["sql"], "sql", theme="monokai", word_wrap=True))

                # ── Save as DataFrame (needed for chart) ──────────
                validated = RunSqlInput(**args)
                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                except Exception as e:
                    result = f"SQL_ERROR: {str(e)}"
                    last_df = None

                preview = "\n".join(result.split("\n")[:3])
                console.print(f"[dim]  ↳ {preview}[/]\n")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })

    # ── Final structured answer ───────────────────────────────────
    final = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=messages + [{
            "role": "user",
            "content": "Now give your final structured answer including plotly_code."
        }],
        response_format=AgentAnswer
    )

    result = final.choices[0].message.parsed

    # ── Execute agent-written Plotly code ─────────────────────────
    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        console.print(Panel(
            Syntax(result.plotly_code, "python", theme="monokai"),
            title="[bold yellow]Agent-Generated Chart Code", border_style="yellow"
        ))
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            console.print(f"[red]Chart error: {e}[/]")
    else:
        console.print("[dim]No chart for this question.[/]")

    return result

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

def run_agent_with_chart(question: str) -> AgentAnswer:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question}
    ]

    print(f"\nQ: {question}")
    print("-" * 40)

    step = 0
    last_df = None

    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "get_schema":
                print(f"Step {step}: get_schema()")
                result = execute_tool("get_schema", {})

            elif tc.function.name == "run_sql":
                validated = RunSqlInput(**args)
                print(f"Step {step}: run_sql()")
                print(f"  {validated.sql}")
                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                except Exception as e:
                    result = f"SQL_ERROR: {str(e)}"
                    last_df = None

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })

    # ── Final structured answer ───────────────────────────────────
    final = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=messages + [{
            "role": "user",
            "content": "Now give your final structured answer including plotly_code."
        }],
        response_format=AgentAnswer
    )
    result = final.choices[0].message.parsed

    # ── Display ───────────────────────────────────────────────────
    print(f"\nAnswer: {result.answer}")
    print(f"Chart:  {result.chart_type}")
    print(f"Code:  {result.plotly_code}")
    # ── Render chart ──────────────────────────────────────────────
    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            print(f"Chart error: {e}")

    return result

In [ ]:
# ── Tests — try all chart types ───────────────────────────────────
run_agent_with_chart("which gender has best and worst credit history?")

In [ ]:
# ── Conversation memory — persists across questions ───────────────
conversation_history = []

def run_agent_with_memory(question: str) -> AgentAnswer:
    global conversation_history

    # Add new question to existing history
    conversation_history.append({"role": "user", "content": question})

    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history

    print(f"\nQ: {question}")
    print("-" * 40)

    step = 0
    last_df = None

    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        conversation_history.append(msg)  # ← save to memory

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "get_schema":
                print(f"Step {step}: get_schema()")
                result = execute_tool("get_schema", {})

            elif tc.function.name == "run_sql":
                validated = RunSqlInput(**args)
                print(f"Step {step}: run_sql()")
                print(f"  {validated.sql}")
                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                except Exception as e:
                    result = f"SQL_ERROR: {str(e)}"
                    last_df = None

            tool_result = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_result)
            conversation_history.append(tool_result)  # ← save to memory

    # ── Final structured answer ───────────────────────────────────
    final = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=messages + [{
            "role": "user",
            "content": "Now give your final structured answer including plotly_code."
        }],
        response_format=AgentAnswer
    )
    result = final.choices[0].message.parsed

    # Save final answer to memory as assistant message
    conversation_history.append({"role": "assistant", "content": result.answer})

    # ── Display ───────────────────────────────────────────────────
    print(f"\nAnswer: {result.answer}")
    print(f"Chart:  {result.chart_type}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            print(f"Chart error: {e}")

    return result


def reset_memory():
    global conversation_history
    conversation_history = []
    print("Memory cleared.")

In [ ]:
# ── Test — ask follow-up questions ───────────────────────────────
run_agent_with_memory("What is the distribution of loan purposes?")

In [ ]:
# Agent remembers the previous question — no need to repeat context
run_agent_with_memory("Which of those purposes has the highest average credit amount?")

In [ ]:
run_agent_with_memory("whats the average loan tenure")

In [ ]:
def safe_sql_result(df: pd.DataFrame) -> str:
    """
    Never send raw rows to OpenAI.
    Send only shape + stats — enough to reason, nothing sensitive.
    """
    summary = []
    summary.append(f"rows: {len(df)}, columns: {list(df.columns)}")

    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            summary.append(
                f"{col}: min={df[col].min()}, max={df[col].max()}, "
                f"mean={df[col].mean():.2f}, nulls={df[col].isna().sum()}"
            )
        else:
            top = df[col].value_counts().head(5).to_dict()
            summary.append(f"{col}: top_values={top}, nulls={df[col].isna().sum()}")

    return "\n".join(summary)

In [ ]:
MAX_RETRIES = 3  # prevent infinite loops
# model = "gpt-5-mini"

def run_agent_with_error_handling(question: str) -> AgentAnswer:
    global conversation_history

    conversation_history.append({"role": "user", "content": question})
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + conversation_history

    print(f"\nQ: {question}")
    print("-" * 40)

    step = 0
    retries = 0
    last_df = None

    while True:
        response = client.chat.completions.create(
            model="gpt-5-mini",
            messages=messages,
            tools=TOOLS
        )

        msg = response.choices[0].message
        messages.append(msg)
        conversation_history.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            step += 1
            args = json.loads(tc.function.arguments)

            if tc.function.name == "get_schema":
                print(f"Step {step}: get_schema()")
                result = execute_tool("get_schema", {})

            elif tc.function.name == "run_sql":
                print(f"Step {step}: run_sql()")

                # ── Pydantic validation ───────────────────────────
                try:
                    validated = RunSqlInput(**args)
                except Exception as e:
                    result = f"VALIDATION_ERROR: {str(e)}"
                    print(f"  Validation failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    # feed error back — agent will self correct
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                    conversation_history.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                    continue

                print(f"  {validated.sql}")

                # ── SQL execution ─────────────────────────────────
                try:
                    last_df = conn.execute(validated.sql).fetchdf()
                    result = last_df.to_string(index=False)
                    retries = 0  # reset retries on success
                except Exception as e:
                    result = f"SQL_ERROR: {str(e)}"
                    print(f"  SQL failed: {e}")
                    retries += 1
                    if retries >= MAX_RETRIES:
                        print("  Max retries reached. Stopping.")
                        return None
                    # feed error back — agent will self correct
                    print(f"  Retry {retries}/{MAX_RETRIES} — feeding error back to agent")

            tool_result = {"role": "tool", "tool_call_id": tc.id, "content": result}
            messages.append(tool_result)
            conversation_history.append(tool_result)

    # ── Final structured answer ───────────────────────────────────
    try:
        df_columns = list(last_df.columns) if last_df is not None else []

        final = client.beta.chat.completions.parse(
            model="gpt-5.4",
            messages=messages + [{
                "role": "user",
                "content": f"The final dataframe has these exact columns: {df_columns}. Now give your final structured answer and make sure plotly_code only uses these column names."
            }],
            response_format=AgentAnswer
        )
        result = final.choices[0].message.parsed
    except Exception as e:
        print(f"Final answer error: {e}")
        return None

    conversation_history.append({"role": "assistant", "content": result.answer})

    # ── Display ───────────────────────────────────────────────────
    print(f"\nAnswer: {result.answer}")
    print(f"Chart:  {result.chart_type}")

    if result.plotly_code and result.chart_type != "none" and last_df is not None:
        try:
            local_vars = {"df": last_df, "px": px, "go": go}
            exec(result.plotly_code, local_vars)
            local_vars["fig"].show()
        except Exception as e:
            print(f"Chart error: {e}")

    return result

In [ ]:
# ── Test normal question ──────────────────────────────────────────
reset_memory()

run_agent_with_error_handling("distribution of default wrt gender & age")

In [ ]:
run_agent_with_error_handling("lets look into the 46-55 bucket and its credit instrument type")

In [ ]:
# Cell 2 — all the code
import json
import duckdb
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import os

_ = load_dotenv()

# ── Config ──────────────────────────────────────────────
CSV_PATH = "../data/credit.csv"  # ← change this
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# ── DB ───────────────────────────────────────────────────
conn = duckdb.connect(":memory:")
conn.execute(f"CREATE TABLE data AS SELECT * FROM read_csv_auto('{CSV_PATH}')")
# print(conn.execute("DESCRIBE data").fetchdf())

# ── Models ───────────────────────────────────────────────
class RunSqlInput(BaseModel):
    sql: str

class AgentAnswer(BaseModel):
    answer: str
    sql_used: str
    chart_type: Literal["bar", "line", "pie", "table", "none"]


In [ ]:
# ── Agent ────────────────────────────────────────────────
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_schema",
            "description": "Get table columns and types. Call this first.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_sql",
            "description": "Run SQL on the 'data' table.",
            "parameters": {
                "type": "object",
                "properties": {"sql": {"type": "string"}},
                "required": ["sql"]
            }
        }
    }
]

def ask(question: str) -> AgentAnswer:
    messages = [
        {"role": "system", "content": "You are a data analyst. Table is called 'data'. Call get_schema first, then run SQL."},
        {"role": "user", "content": question}
    ]

    while True:
        response = client.chat.completions.create(model="gpt-4o", messages=messages, tools=TOOLS)
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            if tc.function.name == "get_schema":
                schema = conn.execute("DESCRIBE data").fetchdf().to_string()
                print(f"→ get_schema")
                result = schema
            elif tc.function.name == "run_sql":
                args = RunSqlInput(**json.loads(tc.function.arguments))
                print(f"→ run_sql: {args.sql}")
                try:
                    result = conn.execute(args.sql).fetchdf().to_string()
                except Exception as e:
                    result = f"ERROR: {e}"

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    structured = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=messages + [{"role": "user", "content": "Give your final structured answer."}],
        response_format=AgentAnswer,
    )
    return structured.choices[0].message.parsed

In [ ]:
result = ask("How many rows are there?")
print(result)

In [ ]:
import tiktoken

encoder = tiktoken.encoding_for_model("gpt-4o")

def count_tokens(messages: list) -> int:
    total = 0
    for m in messages:
        if isinstance(m, dict) and isinstance(m.get("content"), str):
            total += len(encoder.encode(m["content"]))
    return total

# ── Test ──────────────────────────────────────────────────────────
test_messages = [
    {"role": "user", "content": "show me top 5 products"},
    {"role": "assistant", "content": "here are the top 5 products..."},
]

print(f"tokens: {count_tokens(test_messages)}")

In [ ]:
def get_user_turn_indices(history: list) -> list:
    """
    Returns the index of each user message in history.
    Each index = start of a new turn.
    """
    return [
        i for i, m in enumerate(history)
        if isinstance(m, dict) and m.get("role") == "user"
    ]

# ── Test ──────────────────────────────────────────────────────────
fake_history = [
    {"role": "user",      "content": "question 1"},
    {"role": "assistant", "content": "answer 1"},
    {"role": "tool",      "content": "sql result"},
    {"role": "user",      "content": "question 2"},
    {"role": "assistant", "content": "answer 2"},
    {"role": "user",      "content": "question 3"},
    {"role": "assistant", "content": "answer 3"},
]

indices = get_user_turn_indices(fake_history)
print(f"user turn indices: {indices}")
# expected: [0, 3, 5]

# ── Why this matters ──────────────────────────────────────────────
# to keep last 2 turns:
#   indices[-2] = 3  ← cut here
#   keep history[3:]
# to summarise first 2 turns:
#   indices[2] = 5   ← cut here
#   compress history[:5]

keep_last_2 = fake_history[indices[-2]:]
print(f"\nkeeping last 2 turns ({len(keep_last_2)} messages):")
for m in keep_last_2:
    print(f"  {m['role']}: {m['content']}")

In [ ]:
def summarise_turns(turns: list) -> str:
    text = "\n".join(
        f"{m['role']}: {m['content'][:400]}"
        for m in turns
        if isinstance(m, dict) and isinstance(m.get("content"), str)
    )

    print(f"  summarising {len(turns)} messages...")

    response = client.chat.completions.create(
        model=AGENT_MODEL,
        messages=[{
            "role": "user",
            "content": f"""Summarise this conversation history concisely.
Keep: questions asked, SQL findings, key numbers, insights found.
4-5 sentences max.

{text}"""
        }]
    )

    return response.choices[0].message.content

    return response.choices[0].message.content


# ── Test ──────────────────────────────────────────────────────────
fake_turns = [
    {"role": "user",      "content": "how many products are there?"},
    {"role": "tool",      "content": "COUNT(*) = 504"},
    {"role": "assistant", "content": "there are 504 products in the database"},
    {"role": "user",      "content": "which product has most transactions?"},
    {"role": "tool",      "content": "ProductID=870, Name=Tube, count=3688"},
    {"role": "assistant", "content": "Tube (ID 870) has the most transactions at 3688"},
]

summary = summarise_turns(fake_turns)
print(f"\nSummary:\n{summary}")
